In [0]:
import sys
import os
import json

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from modules.utils.date import get_target_yyyymm
from modules.transformation.metadata import add_processed_timestamp
from pyspark.sql.functions import lit

In [0]:
# Get last downloaded bike trips data of given city
# 1. Construct volume path based on city/system and current month.
# 2. Load trips CSV data for target month and add city column.
# 3. Add processed timestamp metadata.
# 4. Append data to bronze table for the city.
months_ago = int(dbutils.widgets.get("months_ago"))
target_month = get_target_yyyymm(months_ago=months_ago)
city = dbutils.widgets.get("city")
city_dict = json.loads(dbutils.widgets.get(city))

volume_path: str = f"/Volumes/bikes/00_landing/data_sources/{city_dict["system"]}/trips/{target_month}"

In [0]:
df = spark.read.format("csv").option("header",True).load(volume_path).\
        withColumn("city", lit(city_dict["city"]))

In [0]:
df = add_processed_timestamp(df)

In [0]:
df.write.mode("append").saveAsTable(f"bikes.01_bronze.{city_dict["city_id"]}_trips_raw")